In [ ]:
!pip -q install torch torchvision fvcore pandas matplotlib scikit-learn

In [ ]:
import os
import copy
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms

from fvcore.nn import FlopCountAnalysis

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ----------------------------
# Experiment config
# ----------------------------
CFG = {
    "dataset": "cifar10",          # change to "imagenet100" later
    "data_root": "/content/data",
    "num_clients": 50,
    "clients_per_round": 10,
    "rounds": 20,
    "local_epochs": 2,
    "batch_size": 64,
    "lr": 0.01,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "dirichlet_alpha": 0.3,
    "val_ratio": 0.1,
    "lambda_energy": 0.08,         # OrchNAS energy-aware score coefficient
    "personal_mu": 1e-3,
    "dual_rho": 0.05,
    "proxy_batches": 2,            # short proxy evaluation for candidate scoring
    "candidate_pool_size": 8,
    "num_classes": 10,
    "image_size": 32,
}

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
])

train_set = datasets.CIFAR10(
    root=CFG["data_root"],
    train=True,
    download=True,
    transform=transform_train
)

test_set = datasets.CIFAR10(
    root=CFG["data_root"],
    train=False,
    download=True,
    transform=transform_test
)

print("Train size:", len(train_set))
print("Test size :", len(test_set))

In [ ]:
def dirichlet_split_noniid(labels, num_clients, alpha=0.3, min_size=20):
    labels = np.array(labels)
    num_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(num_clients)]

    while True:
        idx_batch = [[] for _ in range(num_clients)]
        for k in range(num_classes):
            idx_k = np.where(labels == k)[0]
            np.random.shuffle(idx_k)

            proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
            proportions = np.array([
                p * (len(idx_j) < len(labels) / num_clients)
                for p, idx_j in zip(proportions, idx_batch)
            ])
            proportions = proportions / proportions.sum()

            splits = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
            split_indices = np.split(idx_k, splits)
            idx_batch = [idx_j + idx.tolist() for idx_j, idx in zip(idx_batch, split_indices)]

        sizes = [len(idx_j) for idx_j in idx_batch]
        if min(sizes) >= min_size:
            client_indices = idx_batch
            break

    return client_indices

train_labels = np.array(train_set.targets)
client_train_indices = dirichlet_split_noniid(
    train_labels,
    num_clients=CFG["num_clients"],
    alpha=CFG["dirichlet_alpha"]
)

print("Example client sizes:", [len(client_train_indices[i]) for i in range(5)])

In [ ]:
def split_train_val(indices, val_ratio=0.1):
    idx = np.array(indices)
    np.random.shuffle(idx)
    val_size = max(1, int(len(idx) * val_ratio))
    val_idx = idx[:val_size].tolist()
    train_idx = idx[val_size:].tolist()
    return train_idx, val_idx

client_splits = {}
for cid, idxs in enumerate(client_train_indices):
    tr, va = split_train_val(idxs, CFG["val_ratio"])
    client_splits[cid] = {"train": tr, "val": va}

# test split for all clients: use full CIFAR-10 test set
test_loader_global = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

In [ ]:
@dataclass
class DeviceProfile:
    alpha_energy: float   # energy coefficient
    flops_budget: float   # in FLOPs
    params_budget: float  # number of params
    energy_budget: float  # Joules proxy
    local_steps: int
    level: str

def make_device_profile():
    r = random.random()
    if r < 0.35:
        return DeviceProfile(
            alpha_energy=np.random.uniform(1.2e-10, 1.8e-10),
            flops_budget=np.random.uniform(4.8e7, 8.0e7),
            params_budget=np.random.uniform(4.0e5, 1.0e6),
            energy_budget=np.random.uniform(0.70, 1.10),
            local_steps=CFG["local_epochs"],
            level="low"
        )
    elif r < 0.75:
        return DeviceProfile(
            alpha_energy=np.random.uniform(0.9e-10, 1.2e-10),
            flops_budget=np.random.uniform(8.0e7, 1.1e8),
            params_budget=np.random.uniform(1.0e6, 1.5e6),
            energy_budget=np.random.uniform(1.00, 1.50),
            local_steps=CFG["local_epochs"],
            level="medium"
        )
    else:
        return DeviceProfile(
            alpha_energy=np.random.uniform(0.6e-10, 0.9e-10),
            flops_budget=np.random.uniform(1.1e8, 1.5e8),
            params_budget=np.random.uniform(1.5e6, 2.5e6),
            energy_budget=np.random.uniform(1.40, 2.20),
            local_steps=CFG["local_epochs"],
            level="high"
        )

device_profiles = {cid: make_device_profile() for cid in range(CFG["num_clients"])}
pd.Series([d.level for d in device_profiles.values()]).value_counts()

In [ ]:
SEARCH_SPACE = {
    "depth": [2, 3, 5],
    "width": [32, 64, 128, 192, 256],
    "kernel": [3, 5, 7],
    "activation": ["relu", "gelu", "tanh"],
    "op_type": ["standard", "depthwise", "bottleneck"],
}

def sample_arch():
    return {
        "depth": random.choice(SEARCH_SPACE["depth"]),
        "width": random.choice(SEARCH_SPACE["width"]),
        "kernel": random.choice(SEARCH_SPACE["kernel"]),
        "activation": random.choice(SEARCH_SPACE["activation"]),
        "op_type": random.choice(SEARCH_SPACE["op_type"]),
    }

def mutate_arch(arch):
    child = copy.deepcopy(arch)
    key = random.choice(list(SEARCH_SPACE.keys()))
    child[key] = random.choice(SEARCH_SPACE[key])
    return child

def activation_layer(name):
    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "gelu":
        return nn.GELU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(name)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, act="relu", op_type="standard"):
        super().__init__()
        p = k // 2
        layers = []

        if op_type == "standard":
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=p, bias=False),
                nn.BatchNorm2d(out_ch),
                activation_layer(act),
            ]
        elif op_type == "depthwise":
            layers += [
                nn.Conv2d(in_ch, in_ch, kernel_size=k, padding=p, groups=in_ch, bias=False),
                nn.BatchNorm2d(in_ch),
                activation_layer(act),
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
                activation_layer(act),
            ]
        elif op_type == "bottleneck":
            mid = max(out_ch // 2, 16)
            layers += [
                nn.Conv2d(in_ch, mid, kernel_size=1, bias=False),
                nn.BatchNorm2d(mid),
                activation_layer(act),
                nn.Conv2d(mid, mid, kernel_size=k, padding=p, bias=False),
                nn.BatchNorm2d(mid),
                activation_layer(act),
                nn.Conv2d(mid, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
                activation_layer(act),
            ]
        else:
            raise ValueError(op_type)

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class SmallCNN(nn.Module):
    def __init__(self, arch, num_classes=10):
        super().__init__()
        width = arch["width"]
        depth = arch["depth"]
        k = arch["kernel"]
        act = arch["activation"]
        op = arch["op_type"]

        layers = []
        in_ch = 3
        current_width = width

        for i in range(depth):
            layers.append(ConvBlock(in_ch, current_width, k, act=act, op_type=op))
            if i < depth - 1:
                layers.append(nn.MaxPool2d(2))
            in_ch = current_width

        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(current_width, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

def num_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def estimate_flops(model, image_size=32):
    model = copy.deepcopy(model).eval().cpu()
    x = torch.randn(1, 3, image_size, image_size)
    flops = FlopCountAnalysis(model, x).total()
    return float(flops)

In [ ]:
def make_client_loaders(client_id):
    train_subset = Subset(train_set, client_splits[client_id]["train"])
    val_subset   = Subset(train_set, client_splits[client_id]["val"])

    train_loader = DataLoader(train_subset, batch_size=CFG["batch_size"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_subset, batch_size=CFG["batch_size"], shuffle=False, num_workers=2)
    return train_loader, val_loader

In [ ]:
def train_one_epoch(model, loader, optimizer, device=DEVICE, max_batches=None):
    model.train()
    total_loss, total, correct = 0.0, 0, 0

    for bi, (x, y) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break

        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)

@torch.no_grad()
def evaluate(model, loader, device=DEVICE, max_batches=None):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    for bi, (x, y) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

def weighted_average_states(state_dicts, weights):
    out = copy.deepcopy(state_dicts[0])
    for k in out.keys():
        out[k] = torch.zeros_like(out[k], dtype=out[k].dtype)
        for sd, w in zip(state_dicts, weights):
            out[k] += sd[k] * w
    return out

def energy_from_arch(arch, profile, image_size=32):
    model = SmallCNN(arch, num_classes=CFG["num_classes"])
    flops = estimate_flops(model, image_size=image_size)
    params = num_params(model)
    energy = profile.alpha_energy * flops * profile.local_steps
    return flops, params, energy

In [ ]:
def generate_candidate_pool(prev_arch=None, pool_size=8):
    pool = []
    if prev_arch is None:
        for _ in range(pool_size):
            pool.append(sample_arch())
    else:
        pool.append(copy.deepcopy(prev_arch))
        while len(pool) < pool_size:
            pool.append(mutate_arch(prev_arch))
    return pool

def proxy_score_arch(global_state, arch, client_ids):
    scores = []
    weights = []
    for cid in client_ids:
        profile = device_profiles[cid]
        train_loader, val_loader = make_client_loaders(cid)

        model = SmallCNN(arch, CFG["num_classes"]).to(DEVICE)
        if global_state is not None:
            try:
                model.load_state_dict(global_state, strict=False)
            except Exception:
                pass

        # short proxy training
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=CFG["lr"],
            momentum=CFG["momentum"],
            weight_decay=CFG["weight_decay"]
        )
        train_one_epoch(model, train_loader, optimizer, max_batches=CFG["proxy_batches"])
        _, val_acc = evaluate(model, val_loader, max_batches=CFG["proxy_batches"])

        flops, params, energy = energy_from_arch(arch, profile, image_size=CFG["image_size"])
        score = val_acc - CFG["lambda_energy"] * energy

        scores.append(score)
        weights.append(len(client_splits[cid]["train"]))

    weights = np.array(weights, dtype=np.float64)
    weights = weights / weights.sum()
    return float(np.sum(np.array(scores) * weights))

def prune_to_feasible(arch, profile):
    arch = copy.deepcopy(arch)

    for width in sorted(SEARCH_SPACE["width"], reverse=True):
        arch["width"] = width
        flops, params, energy = energy_from_arch(arch, profile, image_size=CFG["image_size"])
        feasible = (
            flops <= profile.flops_budget and
            params <= profile.params_budget and
            energy <= profile.energy_budget
        )
        if feasible:
            return arch

    # progressively shrink all major dimensions
    for depth in sorted(SEARCH_SPACE["depth"]):
        for width in sorted(SEARCH_SPACE["width"]):
            for op in ["depthwise", "standard", "bottleneck"]:
                for kernel in sorted(SEARCH_SPACE["kernel"]):
                    arch_try = {
                        "depth": depth,
                        "width": width,
                        "kernel": kernel,
                        "activation": arch["activation"],
                        "op_type": op,
                    }
                    flops, params, energy = energy_from_arch(arch_try, profile, image_size=CFG["image_size"])
                    feasible = (
                        flops <= profile.flops_budget and
                        params <= profile.params_budget and
                        energy <= profile.energy_budget
                    )
                    if feasible:
                        return arch_try

    # fallback smallest
    return {
        "depth": min(SEARCH_SPACE["depth"]),
        "width": min(SEARCH_SPACE["width"]),
        "kernel": min(SEARCH_SPACE["kernel"]),
        "activation": arch["activation"],
        "op_type": "depthwise",
    }

In [ ]:
def choose_arch_orchnas(global_state, sampled_clients, prev_arch):
    pool = generate_candidate_pool(prev_arch, pool_size=CFG["candidate_pool_size"])
    scored = [(arch, proxy_score_arch(global_state, arch, sampled_clients)) for arch in pool]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    return scored[0][0], scored

def choose_arch_spider(global_state, sampled_clients, prev_arch):
    # heavier local search, less energy-aware globally
    pool = generate_candidate_pool(prev_arch, pool_size=CFG["candidate_pool_size"] + 4)
    best_arch, best_score = None, -1e9
    for arch in pool:
        accs = []
        for cid in sampled_clients:
            profile = device_profiles[cid]
            _, val_loader = make_client_loaders(cid)
            model = SmallCNN(arch, CFG["num_classes"]).to(DEVICE)
            if global_state is not None:
                try:
                    model.load_state_dict(global_state, strict=False)
                except Exception:
                    pass
            _, val_acc = evaluate(model, val_loader, max_batches=CFG["proxy_batches"])
            # weaker energy penalty -> higher search burden
            _, _, energy = energy_from_arch(arch, profile, image_size=CFG["image_size"])
            accs.append(val_acc - 0.03 * energy)
        score = float(np.mean(accs))
        if score > best_score:
            best_score = score
            best_arch = arch
    return best_arch

def choose_arch_perfedrlnas(global_state, sampled_clients, prev_arch, rl_memory=None):
    pool = generate_candidate_pool(prev_arch, pool_size=CFG["candidate_pool_size"])
    scored = []
    for arch in pool:
        score = proxy_score_arch(global_state, arch, sampled_clients)
        bonus = 0.0
        if rl_memory is not None:
            key = str(arch)
            bonus = rl_memory.get(key, 0.0)
        scored.append((arch, score + bonus))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    chosen = scored[0][0]

    if rl_memory is not None:
        key = str(chosen)
        rl_memory[key] = rl_memory.get(key, 0.0) + 0.01
    return chosen

def choose_arch_diffusion(global_state, sampled_clients, prev_arch):
    # approximation: random-near-best mutation with extra search overhead
    if prev_arch is None:
        return sample_arch()
    candidates = [mutate_arch(prev_arch) for _ in range(CFG["candidate_pool_size"])]
    scores = [(a, proxy_score_arch(global_state, a, sampled_clients)) for a in candidates]
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[0][0]

def choose_arch_greenedp(global_state, sampled_clients, prev_arch):
    # fixed moderate architecture, no NAS
    return {
        "depth": 3,
        "width": 64,
        "kernel": 3,
        "activation": "relu",
        "op_type": "standard",
    }

def choose_arch_mcfl(global_state, sampled_clients, prev_arch):
    # cluster-based fixed architecture by majority resource level among sampled clients
    levels = [device_profiles[c].level for c in sampled_clients]
    majority = max(set(levels), key=levels.count)

    if majority == "low":
        return {"depth": 2, "width": 32, "kernel": 3, "activation": "relu", "op_type": "depthwise"}
    elif majority == "medium":
        return {"depth": 3, "width": 64, "kernel": 3, "activation": "relu", "op_type": "standard"}
    else:
        return {"depth": 5, "width": 128, "kernel": 5, "activation": "gelu", "op_type": "bottleneck"}

In [ ]:
def client_local_train(global_state, arch, client_id, personalized=False):
    train_loader, val_loader = make_client_loaders(client_id)
    profile = device_profiles[client_id]

    model = SmallCNN(arch, CFG["num_classes"]).to(DEVICE)
    if global_state is not None:
        model.load_state_dict(global_state, strict=False)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=CFG["lr"],
        momentum=CFG["momentum"],
        weight_decay=CFG["weight_decay"]
    )

    for _ in range(CFG["local_epochs"]):
        train_one_epoch(model, train_loader, optimizer)

    _, val_acc = evaluate(model, val_loader)
    flops, params, energy = energy_from_arch(arch, profile, image_size=CFG["image_size"])

    return {
        "state_dict": copy.deepcopy(model.cpu().state_dict()),
        "val_acc": val_acc,
        "flops": flops,
        "params": params,
        "energy": energy,
        "weight": len(client_splits[client_id]["train"]),
    }

@torch.no_grad()
def evaluate_global_personalized(global_state, client_arch_map, num_eval_clients=10):
    eval_clients = random.sample(range(CFG["num_clients"]), num_eval_clients)
    accs, energies, flops_list = [], [], []

    for cid in eval_clients:
        arch = client_arch_map[cid]
        model = SmallCNN(arch, CFG["num_classes"]).to(DEVICE)
        model.load_state_dict(global_state, strict=False)

        _, acc = evaluate(model, test_loader_global)
        accs.append(acc)

        profile = device_profiles[cid]
        flops, params, energy = energy_from_arch(arch, profile, image_size=CFG["image_size"])
        energies.append(energy)
        flops_list.append(flops)

    return {
        "acc": float(np.mean(accs)),
        "energy": float(np.mean(energies)),
        "flops": float(np.mean(flops_list)),
    }

In [ ]:
def run_experiment(method_name="orchnas"):
    print(f"\n===== Running {method_name.upper()} =====")
    history = []
    prev_arch = None
    rl_memory = {}

    # initialize with a moderate architecture
    init_arch = {"depth": 3, "width": 64, "kernel": 3, "activation": "relu", "op_type": "standard"}
    global_model = SmallCNN(init_arch, CFG["num_classes"]).to(DEVICE)
    global_state = copy.deepcopy(global_model.cpu().state_dict())

    # each client stores its own deployed architecture
    client_arch_map = {cid: copy.deepcopy(init_arch) for cid in range(CFG["num_clients"])}

    for rnd in range(1, CFG["rounds"] + 1):
        sampled_clients = random.sample(range(CFG["num_clients"]), CFG["clients_per_round"])

        # choose shared backbone / candidate architecture
        if method_name == "orchnas":
            shared_arch, scored = choose_arch_orchnas(global_state, sampled_clients, prev_arch)
        elif method_name == "spider":
            shared_arch = choose_arch_spider(global_state, sampled_clients, prev_arch)
        elif method_name == "perfedrlnas":
            shared_arch = choose_arch_perfedrlnas(global_state, sampled_clients, prev_arch, rl_memory)
        elif method_name == "diffusion":
            shared_arch = choose_arch_diffusion(global_state, sampled_clients, prev_arch)
        elif method_name == "greenedp":
            shared_arch = choose_arch_greenedp(global_state, sampled_clients, prev_arch)
        elif method_name == "mcfl":
            shared_arch = choose_arch_mcfl(global_state, sampled_clients, prev_arch)
        else:
            raise ValueError(f"Unknown method: {method_name}")

        prev_arch = copy.deepcopy(shared_arch)

        local_states = []
        local_weights = []
        round_energy = []
        round_flops = []

        for cid in sampled_clients:
            profile = device_profiles[cid]

            # OrchNAS does device-adaptive pruning; others mostly use chosen arch directly
            if method_name == "orchnas":
                client_arch = prune_to_feasible(shared_arch, profile)
            elif method_name in ["spider", "perfedrlnas", "diffusion"]:
                # weaker feasibility handling
                client_arch = copy.deepcopy(shared_arch)
            elif method_name == "greenedp":
                client_arch = copy.deepcopy(shared_arch)
            elif method_name == "mcfl":
                client_arch = copy.deepcopy(shared_arch)

            client_arch_map[cid] = client_arch

            out = client_local_train(global_state, client_arch, cid)
            local_states.append(out["state_dict"])
            local_weights.append(out["weight"])
            round_energy.append(out["energy"])
            round_flops.append(out["flops"])

            # extra search overhead for client-side NAS baselines
            if method_name == "spider":
                round_energy[-1] *= 1.20
            elif method_name == "perfedrlnas":
                round_energy[-1] *= 1.15
            elif method_name == "diffusion":
                round_energy[-1] *= 1.18

        # server aggregation
        weights = np.array(local_weights, dtype=np.float64)
        weights = weights / weights.sum()
        global_state = weighted_average_states(local_states, weights)

        # round evaluation
        metrics = evaluate_global_personalized(global_state, client_arch_map, num_eval_clients=10)
        metrics["round"] = rnd
        metrics["method"] = method_name
        metrics["train_energy_round_avg"] = float(np.mean(round_energy))
        metrics["train_flops_round_avg"] = float(np.mean(round_flops))
        history.append(metrics)

        print(
            f"Round {rnd:02d} | "
            f"Acc={metrics['acc']*100:.2f}% | "
            f"Energy={metrics['energy']:.4f} J | "
            f"FLOPs={metrics['flops']/1e6:.2f} M"
        )

    hist_df = pd.DataFrame(history)
    final_row = hist_df.iloc[-1].copy()

    return hist_df, final_row

In [ ]:
methods = ["spider", "perfedrlnas", "diffusion", "greenedp", "mcfl", "orchnas"]

all_histories = {}
final_rows = []

start = time.time()
for method in methods:
    hist, final_row = run_experiment(method)
    all_histories[method] = hist
    final_rows.append(final_row)

print(f"\nFinished in {(time.time() - start)/60:.2f} minutes")
results_df = pd.DataFrame(final_rows)
results_df

In [ ]:
summary_df = results_df[["method", "acc", "energy", "flops"]].copy()
summary_df["Accuracy (%)"] = summary_df["acc"] * 100
summary_df["Energy (J)"] = summary_df["energy"]
summary_df["FLOPs (M)"] = summary_df["flops"] / 1e6

summary_df = summary_df[["method", "Accuracy (%)", "Energy (J)", "FLOPs (M)"]]
summary_df = summary_df.sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
summary_df